# 第08講　実演：データ分析

最初に紹介した **A さんの少子高齢化の分析** を、
**① データ入手 → ② データクリーニング → ③ グラフ作成** の順に再現する。

使う関数や手法は授業範囲外なので暗記は不要。
「生のデータがグラフになるまでに、どれだけ加工が必要か」を体感することが目的。

## 0　セットアップ

In [ ]:
import piplite
await piplite.install("seaborn")

import pandas as pd
import seaborn as sns
import pyodide
from matplotlib import pyplot as plt
from matplotlib import font_manager as fm
%matplotlib inline

# グラフに日本語（総数・男・女など）を表示するため Noto Sans JP を取得
from pyodide.http import pyfetch
resp = await pyfetch("https://raw.githubusercontent.com/google/fonts/main/ofl/notosansjp/NotoSansJP%5Bwght%5D.ttf")
with open("NotoSansJP.ttf", "wb") as f:
    f.write(await resp.bytes())
fm.fontManager.addfont("NotoSansJP.ttf")
sns.set_theme(style="darkgrid", font="Noto Sans JP")

## 1　データ入手

- **世界の高齢化**：国連・世界銀行の「65 歳以上人口の割合（%）」（2000 年・2010 年）
- **日本の高齢化**：総務省「国勢調査」の年齢別人口（1920〜2020 年）

いずれも整形済みの CSV を GitHub から読み込む。

In [ ]:
URL = "https://raw.githubusercontent.com/ChungWookyung/kg-jupyterlite-data-analysis/main/content/DATA/"

world = pd.read_csv(pyodide.http.open_url(URL + "world_ageing.csv"))
japan = pd.read_csv(pyodide.http.open_url(URL + "japan_ageing.csv"))
world.head()

In [ ]:
japan.head()

## 2　データクリーニング（世界）

`world` は 1 列に国名、2 列に各年の値が並ぶ **横長** の形。
グラフを描きやすいよう、`melt()` で **縦長**（国・年・割合）に変形する。

In [ ]:
world_long = world.melt(id_vars="country", var_name="year", value_name="share")
world_long["year"] = world_long["year"].str.replace("share_", "").astype(int)
world_long.head()

## 3　グラフ作成：世界各国の高齢化（2000 年）

割合の高い順に並べ替え、横棒グラフにする。

In [ ]:
d = world_long[world_long["year"] == 2000].sort_values("share", ascending=False)
sns.barplot(data=d, y="country", x="share", hue="country", palette="Spectral", legend=False)
plt.title("Ageing Population As of 2000")
plt.xlabel("Value"); plt.ylabel("Country or Area")

## 3　グラフ作成：世界各国の高齢化（2010 年）

In [ ]:
d = world_long[world_long["year"] == 2010].sort_values("share", ascending=False)
sns.barplot(data=d, y="country", x="share", hue="country", palette="Spectral", legend=False)
plt.title("Ageing Population As of 2010")
plt.xlabel("Value"); plt.ylabel("Country or Area")

**解釈**：2000 年・2010 年とも **日本が最も高齢化が進んでいる**。
2010 年には日本だけが 20% を大きく超え、世界の中でも突出している。

## 4　データクリーニング（日本）

国勢調査のデータから **高齢化率（65 歳以上人口 ÷ 総人口 × 100）** を計算する。
総数・男・女の 3 系列を作り、`melt()` で縦長に整える。
（男女別の人口が分かるのは 1960 年以降）

In [ ]:
japan["総数"] = japan["pop65"]        / japan["total_pop"] * 100
japan["男"]   = japan["pop65_male"]   / japan["total_pop"] * 100
japan["女"]   = japan["pop65_female"] / japan["total_pop"] * 100

jp_long = japan.melt(id_vars="year", value_vars=["総数", "男", "女"],
                     var_name="分類", value_name="比率(%)")
jp_long = jp_long.rename(columns={"year": "年度"}).dropna()
jp_long.head()

## 5　グラフ作成：日本の高齢化推移（折れ線）

In [ ]:
sns.lineplot(data=jp_long, x="年度", y="比率(%)", hue="分類")

## 5　グラフ作成：日本の高齢化推移（棒グラフ）

In [ ]:
sns.barplot(data=jp_long, x="年度", y="比率(%)", hue="分類")
plt.xticks(rotation=90)

**解釈**：日本の高齢化率は **1970 年代から急激に上昇**し、
2020 年には **28.6%**（約 4 人に 1 人以上が 65 歳以上）に達した。
女性のほうが男性より高齢者人口の割合が高い。

## 要点整理

- データ分析の時間の **8 割はデータクリーニング** に費やされる。
- 生のデータ（横長・人数）は、`melt()` や割合計算などで
  **グラフ向きの形（縦長・%）** に整える必要がある。
- グラフは説明（解釈）とセットで初めて意味を持つ。